# AML - SAIGE-GENE SET-BASED ANALYSIS -  ALLELE COUNT  
This script calculates variant minor allele counts from the genomic PLINK file used to generate the sparse Genetic Relationship Matrix (GRM). Before computing the counts, the PLINK file is subsetted to exclude samples that were not used in constructing the sparse GRM.

In [ ]:
# Load packages 
from datetime import datetime
import pandas as pd 
import os

In [ ]:
# Workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")
# Workspace bucket 
my_bucket

# COMMAND 

In [ ]:
%%writefile microarray_counts.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    # Check if the cohort file for the current population exists
    if [ ! -f "${cohort}/${pop}-like_exome_qc.txt" ]; then
        echo "Cohort file for ${pop} not found!"
        exit 1
    fi
    
    # Copy new plink module into the plink2 
    cp "${plink2}" plink2

    # Grant permision to run plink2 in environment
    chmod +x plink2
    
    # Check number of samples in each cohort 
    echo "Number of samples in ${pop}:"
    wc -l "${cohort}/${pop}-like_exome_qc.txt"
    
    # Run PLINK2 for variant counts for the current population
    ./plink2 --bfile "${genos}/arrays" \
             --keep "${cohort}/${pop}-like_exome_qc.txt" \
             --freq counts \
             --out "${output}/${pop}-like_microarray"
        
done


In [ ]:
# Copy script to my_bucket 
!gsutil cp microarray_counts.sh {my_bucket}/data/scripts/    

In [ ]:
# Copy script to my_bucket 
!gsutil cat {my_bucket}/data/scripts/microarray_counts.sh

# SUMBIT COMMAND 

In [ ]:
%%bash --out microarray_counts 

#############################################################################
# DSUB: SUBMIT SCRIPT FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Source the dsub setup script, which likely contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/microarray/plink"

#############################################################################
# SUBMIT DSUB SCRIPT JOB FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Use a prebuilt container with PLINK2
# Set total disk space required 
# Define logging directory for dsub execution logs
# Input PLINK files (BED, BIM, FAM) from Google Cloud Storage (GCS)
# Provide cohort sample lists (recursively input from GCS)
# Define output directory for variant QC results (recursively stores outputs)
# Set environment variables for the script
# Specify the script to execute within the container

aou_dsub \
  --disk-size 220 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step01/microarray_counts/" \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env output="${WORKSPACE_BUCKET}/data/saige_step01/microarray_counts" \
  --script "${WORKSPACE_BUCKET}/data/scripts/microarray_counts.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################
